# ACHTUNG: In diesem Notebook werden händisch Werte nachgetragen. Am besten Notebook nachvollziehen und mit dem nächsten Datensatz im nächsten Notebook weitermachen!

## Height und Weight cleaning
- Prüfen der NA und 0 Werte in den Height und Weight Spalten
- beide gleiche Anzahl an NAs somit wohl Deckungsgleich

In [2]:
import pandas as pd
import os

In [3]:
df = pd.read_pickle('../../data/processed/13_cleaned_master_data.pkl')

In [7]:
df.isna().sum()

race                                0
year                                0
url                                 0
rank                                0
rider_url                           0
time_gap                            0
birthdate                           0
height                            910
name                                0
nationality                         0
weight                            910
url_name                            0
departure                           0
arrival                             0
distance                            0
vertical_meters                     0
won_how                             0
avg_speed                           0
race_ranking                        0
one_day_races                       0
gc                                  0
time_trial                          0
sprint                              0
climber                             0
hills                               0
stage_nr                            0
date        

Prüfen ob Index heigth und weight immer gleich?

In [10]:
nan_height_idx = df[df['height'].isna()].index
nan_weight_idx = df[df['weight'].isna()].index

indices_are_equal = nan_height_idx.equals(nan_weight_idx)

print(f"Index Größe und Gewicht gleich??  {indices_are_equal}")

Index Größe und Gewicht gleich??  True


Prüfen wie viele Fahrer es tatsächlich betrifft     

In [ ]:

missing_physis_df = df[df['height'].isna()]

anzahl_einzigartige_fahrer = missing_physis_df['rider_url'].nunique()


print(anzahl_einzigartige_fahrer)


16


Nur 16 Fahrer -> händisches Cleaning möglich

In [12]:
missing_physis = df[df['height'].isna()]

recherche_fahrer_df = missing_physis[['name', 'nationality', 'rider_url']].drop_duplicates().copy()


recherche_fahrer_df['height_recherchiert'] = ""
recherche_fahrer_df['weight_recherchiert'] = ""


recherche_fahrer_df = recherche_fahrer_df.sort_values(by='name')


target_dir = "../../data/interim"
os.makedirs(target_dir, exist_ok=True)

target_path = os.path.join(target_dir, "16_fahrer_physis_recherche.csv")
recherche_fahrer_df.to_csv(target_path, index=False, sep=';')


In [14]:
df['height'].head(10)

0    1.92
1    1.83
2    1.71
3    1.76
4    1.74
5    1.81
6    1.79
7    1.76
8    1.80
9    1.72
Name: height, dtype: float64

https://de.wikiital.com -> 7 Fahrerdaten 




In [ ]:
recherche_physis = pd.read_csv('../../data/interim/16_fahrer_physis_recherche.csv', sep=';')


recherche_physis['height_recherchiert'] = pd.to_numeric(recherche_physis['height_recherchiert'], errors='coerce')
recherche_physis['weight_recherchiert'] = pd.to_numeric(recherche_physis['weight_recherchiert'], errors='coerce')


df = df.merge(recherche_physis[['rider_url', 'height_recherchiert', 'weight_recherchiert']], on='rider_url', how='left')


df['height'] = df['height'].combine_first(df['height_recherchiert'])
df['weight'] = df['weight'].combine_first(df['weight_recherchiert'])


df.drop(columns=['height_recherchiert', 'weight_recherchiert'], inplace=True)




rest_missing_rows = df['height'].isna().sum()
rest_missing_riders = df[df['height'].isna()]['rider_url'].nunique()

print(f"Verbleibende Zeilen mit fehlender Physis: {rest_missing_rows}")
print(f"Verbleibende betroffene Fahrer:            {rest_missing_riders}")

Verbleibende Zeilen mit fehlender Physis: 475
Verbleibende betroffene Fahrer:            9


9 verbleibende Fahrer betroffen

- Median Imputation als Lösung

In [ ]:
global_median_height = df['height'].median()
global_median_weight = df['weight'].median()


print(f"Median für Größe (Meter): {global_median_height:.3f} m")
print(f"Median für Gewicht (kg):  {global_median_weight:.1f} kg\n")

# 2. Die verbleibenden Lücken der 9 Fahrer mit diesen Medianen auffüllen
df['height'] = df['height'].fillna(global_median_height)
df['weight'] = df['weight'].fillna(global_median_weight)

# 3. KORREKTUR: Erst runden, dann Datentypen sauber setzen
df['height'] = df['height'].astype(float)

# Für weight: Erst runden, zu float64 machen und dann sicher zu 'Int64' casten
df['weight'] = df['weight'].round(0).astype('float64').astype('Int64')


print(f"Verbleibende NaNs in 'height': {df['height'].isna().sum()} | Datentyp: {df['height'].dtype}")
print(f"Verbleibende NaNs in 'weight': {df['weight'].isna().sum()} | Datentyp: {df['weight'].dtype}")

Median für Größe (Meter): 1.800 m
Median für Gewicht (kg):  68.0 kg

Verbleibende NaNs in 'height': 0 | Datentyp: float64
Verbleibende NaNs in 'weight': 0 | Datentyp: Int64

So sehen die bereinigten Daten jetzt aus:
                    name  height  weight
              Tom Boonen    1.92      82
            Thor Hushovd    1.83      83
           Robbie McEwen    1.71      67
          Stuart O'Grady    1.76      73
Luciano André Pagliarini    1.74      68


In [22]:
# Letzte Prüfung der beiden Werte

height_nans = df['height'].isna().sum()
height_zeros = (df['height'] == 0).sum()


weight_nans = df['weight'].isna().sum()
weight_zeros = (df['weight'] == 0).sum()

print(f"Spalte 'height' (Größe):")
print(f"  - Fehlende Werte (NaN): {height_nans}")
print(f"  - Null-Werte (0):       {height_zeros}\n")

print(f"Spalte 'weight' (Gewicht):")
print(f"  - Fehlende Werte (NaN): {weight_nans}")
print(f"  - Null-Werte (0):       {weight_zeros}")


Spalte 'height' (Größe):
  - Fehlende Werte (NaN): 0
  - Null-Werte (0):       0

Spalte 'weight' (Gewicht):
  - Fehlende Werte (NaN): 0
  - Null-Werte (0):       0


### Checkpoint
Neues .pkl erstellen 14

In [23]:
pfad = '../../data/processed/14_cleaned_master_data.pkl'
df.to_pickle(pfad)